# FPL MaskablePPO Training

Train a MaskablePPO agent across multiple historical FPL seasons.

**Pipeline:**
1. Build multi-season training env (2016-17 to 2022-23)
2. Build eval env (2023-24) for model selection
3. Train MaskablePPO with action masking
4. Evaluate on holdout season (2024-25)
5. Compare against no-op and random baselines

In [ ]:
from pathlib import Path
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

DATA_DIR = Path('../data/raw')
PREDICTOR_MODEL_DIR = Path('../models/point_predictor')  # set to None to skip
RUN_DIR = Path('../runs/fpl_ppo')
RUN_DIR.mkdir(parents=True, exist_ok=True)

RESUME_FROM = Path('../runs/fpl_ppo/final_model/final_model')  # set to None to train from scratch "runs/fpl_ppo/best_model/best_model"

TRAIN_SEASONS = [
    '2016-17', '2017-18', '2018-19', '2019-20',
    '2020-21', '2021-22', '2022-23',
]
EVAL_SEASON = '2023-24'
HOLDOUT_SEASON = '2024-25'

# Set to None to train without the LightGBM predictor
USE_PREDICTOR = PREDICTOR_MODEL_DIR if PREDICTOR_MODEL_DIR.exists() else None
print(f'Predictor: {USE_PREDICTOR or "disabled"}')
print(f'Resume from: {RESUME_FROM or "training from scratch"}')

## 1. Hyperparameters

In [ ]:
TOTAL_TIMESTEPS = 200_000  # ~5,263 episodes
N_ENVS = 10                 # parallel environments (~3-4x speedup)
N_STEPS = 2048             # steps per env per rollout
BATCH_SIZE = 256           # 8 minibatches per epoch
N_EPOCHS = 10
LEARNING_RATE = 3e-4
GAMMA = 0.99               # GW38 discounted to 0.69x
ENT_COEF = 0.02            # mild exploration in large action space
NET_ARCH = [256, 256]
SEED = 42

EVAL_FREQ = 10_000         # evaluate every N total timesteps
N_EVAL_EPISODES = 3
HOLDOUT_EPISODES = 5

print(f'Total timesteps:   {TOTAL_TIMESTEPS:,}')
print(f'Parallel envs:     {N_ENVS}')
print(f'Steps per rollout: {N_STEPS} x {N_ENVS} = {N_STEPS * N_ENVS:,}')
print(f'Batch size:        {BATCH_SIZE}')
print(f'Approx episodes:   ~{TOTAL_TIMESTEPS // 38:,}')

## 2. Build Environments

In [ ]:
from stable_baselines3.common.vec_env import SubprocVecEnv
from fpl_rl.env.fpl_env import FPLEnv

PREDICTION_DATA_DIR = DATA_DIR.parent  # data/ (parent of data/raw/) for id_maps, understat, etc.


def make_train_env(rank, seasons, data_dir, predictor_model_dir, prediction_data_dir):
    """Factory that creates a MultiSeasonFPLEnv in a subprocess."""
    def _init():
        from fpl_rl.training.multi_season_env import MultiSeasonFPLEnv
        env = MultiSeasonFPLEnv(
            seasons=seasons,
            data_dir=data_dir,
            predictor_model_dir=predictor_model_dir,
            prediction_data_dir=prediction_data_dir if predictor_model_dir else None,
            shuffle=True,  # decorrelate parallel envs
        )
        return env
    return _init


print(f'Building {N_ENVS} parallel training envs ({len(TRAIN_SEASONS)} seasons each)...')
print('(Each subprocess loads data independently — this takes a few minutes)')
train_env = SubprocVecEnv([
    make_train_env(i, TRAIN_SEASONS, DATA_DIR, USE_PREDICTOR, PREDICTION_DATA_DIR)
    for i in range(N_ENVS)
])

print(f'\nBuilding eval env ({EVAL_SEASON})...')
eval_kwargs = dict(season=EVAL_SEASON, data_dir=DATA_DIR)
if USE_PREDICTOR is not None:
    from fpl_rl.prediction.integration import PredictionIntegrator
    eval_kwargs['prediction_integrator'] = PredictionIntegrator.from_model(
        USE_PREDICTOR, PREDICTION_DATA_DIR, EVAL_SEASON,
    )
eval_env = FPLEnv(**eval_kwargs)

print(f'\nAction space:      {train_env.action_space}')
print(f'Observation space: {train_env.observation_space.shape}')
print(f'Parallel envs:     {N_ENVS}')
print('Done.')

## 3. Create Model & Callbacks

In [ ]:
from sb3_contrib import MaskablePPO
from fpl_rl.training.callbacks import FPLEvalCallback, FPLEpisodeLogCallback

tb_log_path = str(RUN_DIR / 'tb_logs')

if RESUME_FROM is not None and Path(str(RESUME_FROM) + '.zip').exists():
    model = MaskablePPO.load(
        str(RESUME_FROM),
        env=train_env,
        tensorboard_log=tb_log_path,
        learning_rate=LEARNING_RATE,
        ent_coef=ENT_COEF,
        device='cpu',
    )
    print(f'Resumed from {RESUME_FROM}')
else:
    model = MaskablePPO(
        'MlpPolicy',
        train_env,
        n_steps=N_STEPS,
        batch_size=BATCH_SIZE,
        n_epochs=N_EPOCHS,
        learning_rate=LEARNING_RATE,
        gamma=GAMMA,
        ent_coef=ENT_COEF,
        policy_kwargs=dict(net_arch=NET_ARCH),
        tensorboard_log=tb_log_path,
        seed=SEED,
        verbose=0,
    )
    print('Training from scratch')

best_model_path = RUN_DIR / 'best_model'
best_model_path.mkdir(parents=True, exist_ok=True)

eval_cb = FPLEvalCallback(
    eval_env=eval_env,
    eval_freq=max(EVAL_FREQ // N_ENVS, 1),  # freq is per-env steps with vec env
    n_eval_episodes=N_EVAL_EPISODES,
    best_model_save_path=best_model_path,
    verbose=1,
    show_progress=True,
)
episode_cb = FPLEpisodeLogCallback(verbose=0)

print(f'Model policy: {model.policy}')
n_params = sum(p.numel() for p in model.policy.parameters())
print(f'Parameters:   {n_params:,}')
print(f'Eval freq:    every {EVAL_FREQ:,} total steps ({N_EVAL_EPISODES} episodes)')
print(f'Best model -> {best_model_path}')

## 4. Train

SB3's `progress_bar=True` shows a tqdm bar over total timesteps.

In [ ]:
model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=[eval_cb, episode_cb],
    progress_bar=True,
    reset_num_timesteps=RESUME_FROM is None,
)

print(f'\nTraining complete.')
print(f'Best eval points: {eval_cb.best_mean_points:.1f}')
print(f'Training episodes: {episode_cb._episode_count}')

## 5. Save Final Model

In [ ]:
import datetime

In [ ]:
final_path = RUN_DIR / 'final_model'
final_path.mkdir(parents=True, exist_ok=True)
model.save(str(final_path / 'final_model'))
print(f'Final model saved to {final_path.resolve()}')
print(f'Files: {[f.name for f in final_path.iterdir()]}')
print(f'Model saved at time: {datetime.datetime.now()}')

## 6. Holdout Evaluation (2024-25)

Run the trained agent, a no-op baseline, and a random-masked baseline on the holdout season.

In [ ]:
from tqdm.auto import tqdm

print(f'Building holdout env ({HOLDOUT_SEASON})...')
holdout_kwargs = dict(season=HOLDOUT_SEASON, data_dir=DATA_DIR)
if USE_PREDICTOR is not None:
    holdout_kwargs['prediction_integrator'] = PredictionIntegrator.from_model(
        USE_PREDICTOR, PREDICTION_DATA_DIR, HOLDOUT_SEASON,
    )
holdout_env = FPLEnv(**holdout_kwargs)
print('Done.')

In [ ]:
def run_agent(model, env, n_episodes, desc='Agent'):
    """Run trained agent for n episodes, return list of total points."""
    results = []
    for _ in tqdm(range(n_episodes), desc=desc, unit='ep'):
        obs, _ = env.reset()
        done = False
        pts = 0.0
        while not done:
            masks = env.action_masks()
            action, _ = model.predict(obs, deterministic=True, action_masks=masks)
            obs, _, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            pts = info.get('total_points', pts)
        results.append(pts)
    return results


def run_noop(env, n_episodes):
    """No-op baseline: zero action every GW."""
    results = []
    for _ in tqdm(range(n_episodes), desc='No-op', unit='ep'):
        obs, _ = env.reset()
        done = False
        pts = 0.0
        while not done:
            action = np.zeros(env.action_space.shape, dtype=int)
            obs, _, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            pts = info.get('total_points', pts)
        results.append(pts)
    return results


def run_random(env, n_episodes, seed=42):
    """Random masked baseline: sample valid actions."""
    rng = np.random.default_rng(seed)
    results = []
    for _ in tqdm(range(n_episodes), desc='Random', unit='ep'):
        obs, _ = env.reset()
        done = False
        pts = 0.0
        while not done:
            masks = env.action_masks()
            action = np.zeros(env.action_space.shape, dtype=int)
            offset = 0
            for dim_i, n_choices in enumerate(env.action_space.nvec):
                dim_mask = masks[offset:offset + n_choices]
                valid = np.where(dim_mask)[0]
                if len(valid) > 0:
                    action[dim_i] = rng.choice(valid)
                offset += n_choices
            obs, _, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            pts = info.get('total_points', pts)
        results.append(pts)
    return results

In [ ]:
agent_pts = run_agent(model, holdout_env, HOLDOUT_EPISODES, desc='PPO agent')
noop_pts = run_noop(holdout_env, HOLDOUT_EPISODES)
random_pts = run_random(holdout_env, HOLDOUT_EPISODES)

In [ ]:
def fmt(pts):
    return f'{np.mean(pts):7.1f} +/- {np.std(pts):5.1f}  (min={min(pts):.0f}, max={max(pts):.0f})'

print('=' * 65)
print(f'  Holdout evaluation: {HOLDOUT_SEASON} ({HOLDOUT_EPISODES} episodes)')
print('=' * 65)
print(f"  {'PPO Agent:':<20} {fmt(agent_pts)}")
print(f"  {'No-op baseline:':<20} {fmt(noop_pts)}")
print(f"  {'Random baseline:':<20} {fmt(random_pts)}")
print('=' * 65)

## 7. Training Metrics

Plot key metrics from TensorBoard logs.

In [ ]:
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

# Find the latest TB run
tb_dir = RUN_DIR / 'tb_logs'
latest_run = sorted(tb_dir.iterdir())[-1]
print(f'Loading metrics from: {latest_run.name}')

ea = EventAccumulator(str(latest_run))
ea.Reload()

def get_scalar(tag):
    """Load scalar events, deduplicating by step (keeps last value per step).
    
    This handles the case where multiple event files exist in the same
    TB directory (e.g. from re-running the training cell).
    """
    events = ea.Scalars(tag)
    # Deduplicate: keep the last event for each step
    step_to_value = {}
    for e in events:
        step_to_value[e.step] = e.value
    steps = sorted(step_to_value.keys())
    values = [step_to_value[s] for s in steps]
    return steps, values

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Training Metrics', fontsize=14, fontweight='bold')

# 1. Eval total points
ax = axes[0, 0]
steps, vals = get_scalar('eval/mean_total_points')
_, stds = get_scalar('eval/std_total_points')
ax.plot(steps, vals, 'b-o', markersize=4)
ax.fill_between(steps, np.array(vals) - np.array(stds), np.array(vals) + np.array(stds), alpha=0.2)
ax.set_title('Eval Total Points (2023-24)')
ax.set_xlabel('Timesteps')
ax.set_ylabel('Total Points')
ax.grid(True, alpha=0.3)

# 2. Train episode total points
ax = axes[0, 1]
steps, vals = get_scalar('train/episode_total_points')
ax.plot(steps, vals, 'g-o', markersize=4)
ax.set_title('Train Episode Points')
ax.set_xlabel('Timesteps')
ax.set_ylabel('Total Points')
ax.grid(True, alpha=0.3)

# 3. Value loss
ax = axes[0, 2]
steps, vals = get_scalar('train/value_loss')
ax.plot(steps, vals, 'r-o', markersize=4)
ax.set_title('Value Loss')
ax.set_xlabel('Timesteps')
ax.set_ylabel('Loss')
ax.grid(True, alpha=0.3)

# 4. Policy gradient loss
ax = axes[1, 0]
steps, vals = get_scalar('train/policy_gradient_loss')
ax.plot(steps, vals, 'm-o', markersize=4)
ax.set_title('Policy Gradient Loss')
ax.set_xlabel('Timesteps')
ax.set_ylabel('Loss')
ax.grid(True, alpha=0.3)

# 5. Entropy loss
ax = axes[1, 1]
steps, vals = get_scalar('train/entropy_loss')
ax.plot(steps, vals, 'c-o', markersize=4)
ax.set_title('Entropy Loss')
ax.set_xlabel('Timesteps')
ax.set_ylabel('Entropy')
ax.grid(True, alpha=0.3)

# 6. Explained variance
ax = axes[1, 2]
steps, vals = get_scalar('train/explained_variance')
ax.plot(steps, vals, 'orange', marker='o', markersize=4)
ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax.axhline(y=1, color='k', linestyle='--', alpha=0.3)
ax.set_title('Explained Variance')
ax.set_xlabel('Timesteps')
ax.set_ylabel('Explained Var')
ax.set_ylim(-0.1, 1.1)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## TensorBoard

```bash
tensorboard --logdir runs/fpl_ppo/tb_logs
```

Key metrics:
- `eval/mean_total_points` — held-out season performance
- `train/episode_total_points` — training curve
- `train/entropy_loss` — exploration level

## Loading a Saved Model

```python
from sb3_contrib import MaskablePPO
model = MaskablePPO.load('runs/fpl_ppo/best_model/best_model')
```